## Setup

Scrapes the [Explore Education Statistics data catalogue](https://explore-education-statistics.service.gov.uk/data-catalogue) (API data sets) and maintains a Delta table of all published datasets.

### Pipeline steps

1. Paginate through the catalogue, parsing dataset metadata from each page.
2. Enrich each dataset with the API dataset ID and publication ID from its detail page.
3. Validate the scraped DataFrame (row count, nulls, duplicates).
4. MERGE into the target Delta table (insert / update / delete).
5. Exit with a human-readable change summary (visible in job run output).

### Target table

`catalog_40_copper_statistics_services.analytics_raw.ees_scrape_api_data_catalog`

In [0]:
import json
import math
import re
import time
from datetime import datetime, timezone

import requests
from bs4 import BeautifulSoup
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# ── Source configuration ─────────────────────────────────────
BASE_URL = "https://explore-education-statistics.service.gov.uk"
CATALOGUE_URL = f"{BASE_URL}/data-catalogue"
PER_PAGE = 10  # datasets per catalogue page (set by the EES site)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

# ── Target configuration ────────────────────────────────────
TARGET_TABLE = (
    "catalog_40_copper_statistics_services"
    ".analytics_raw"
    ".ees_scrape_api_data_catalog"
)

# ── Validation thresholds ───────────────────────────────────
REQUIRED_COLUMNS = {"publication", "dataset_name", "dataset_id", "link", "published", "last_updated", "api_dataset_id", "publication_id"}
NON_NULL_COLUMNS = ["dataset_id", "dataset_name", "publication", "published", "last_updated"]
MIN_ROW_COUNT = 10  # safety floor – catalogue should never have fewer than this

# ── Schema evolution (allows new columns to be added via MERGE) ─
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

In [0]:
def fetch_page(page: int) -> BeautifulSoup:
    """Fetch a single catalogue page and return parsed HTML.

    Args:
        page: 1-based page number to fetch.

    Returns:
        Parsed BeautifulSoup document for the requested page.

    Raises:
        requests.HTTPError: If the HTTP request returns a non-2xx status.
    """
    url = f"{CATALOGUE_URL}?dataSetType=api&sortBy=newest&page={page}"
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def get_total_and_pages(soup: BeautifulSoup) -> tuple[int, int]:
    """Extract the total dataset count and page count from a catalogue page.

    Parses the \"X data sets\" text shown in the page header.

    Args:
        soup: Parsed HTML of a catalogue page.

    Returns:
        Tuple of (total_datasets, total_pages).

    Raises:
        ValueError: If the expected \"N data sets\" text is not found in the page.
    """
    match = re.search(r"(\d+)\s+data\s+sets?", soup.get_text())
    if not match:
        raise ValueError(
            "Could not find dataset count on the catalogue page. "
            "The page structure may have changed."
        )
    total = int(match.group(1))
    return total, math.ceil(total / PER_PAGE)


def parse_page(soup: BeautifulSoup) -> list[dict]:
    """Extract dataset records from a single catalogue page.

    Each record contains: publication, dataset_name, dataset_id, link,
    published, and last_updated.

    Args:
        soup: Parsed HTML of a catalogue page.

    Returns:
        List of dataset dicts, one per catalogue entry on the page.
    """
    records = []
    for li in soup.find_all("li", class_="dfe-border-bottom"):
        link_tag = li.find("a", href=re.compile(r"/data-catalogue/data-set/"))
        if not link_tag:
            continue

        spans = link_tag.find_all("span")
        href = link_tag["href"]

        # Extract metadata key-value pairs from the description list
        meta = {}
        dl = li.find("dl")
        if dl:
            for row in dl.find_all("div", class_="govuk-summary-list__row"):
                key = row.find("dt").get_text(strip=True) if row.find("dt") else ""
                val = row.find("dd").get_text(strip=True) if row.find("dd") else ""
                meta[key] = val

        records.append({
            "publication": spans[0].get_text(strip=True) if len(spans) > 0 else None,
            "dataset_name": spans[1].get_text(strip=True) if len(spans) > 1 else None,
            "dataset_id": href.split("/")[-1],
            "link": BASE_URL + href,
            "published": meta.get("Published"),
            "last_updated": meta.get("Last updated"),
        })
    return records


def scrape_catalogue() -> tuple[list[dict], int]:
    """Scrape all pages of the EES data catalogue.

    Paginates through every page, pausing 0.5 s between requests to
    avoid overloading the server.

    Returns:
        Tuple of (scraped_records, expected_total_from_page_header).
    """
    first_soup = fetch_page(1)
    expected_total, total_pages = get_total_and_pages(first_soup)
    print(f"Pages to scrape: {total_pages}")

    all_datasets = parse_page(first_soup)
    print(f"Page 1: {len(all_datasets)} datasets")

    for page in range(2, total_pages + 1):
        time.sleep(0.5)  # polite gap to avoid spamming the pages
        page_data = parse_page(fetch_page(page))
        if not page_data:
            print(f"Page {page}: no results, stopping.")
            break
        all_datasets.extend(page_data)
        print(f"Page {page}: +{len(page_data)} (total: {len(all_datasets)})")

    print(f"\nScraped {len(all_datasets)} datasets.")
    return all_datasets, expected_total

In [0]:
def fetch_api_details(catalogue_id: str) -> dict:
    """Fetch the API dataset ID and publication ID from a dataset's detail page.

    Parses the ``__NEXT_DATA__`` JSON embedded in the page to extract
    ``apiDataSet.id`` and ``publicationSummary.id``.

    Args:
        catalogue_id: The catalogue dataset ID (URL slug on the detail page).

    Returns:
        Dict with ``api_dataset_id`` and ``publication_id`` (None if not found).
    """
    url = f"{BASE_URL}/data-catalogue/data-set/{catalogue_id}"
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")
    script = soup.find("script", id="__NEXT_DATA__")

    result = {"api_dataset_id": None, "publication_id": None}
    if not script:
        return result

    try:
        data = json.loads(script.string)
        page_props = data.get("props", {}).get("pageProps", {})

        api_ds = page_props.get("apiDataSet")
        if api_ds:
            result["api_dataset_id"] = api_ds.get("id")

        pub = page_props.get("publicationSummary")
        if pub:
            result["publication_id"] = pub.get("id")
    except (json.JSONDecodeError, AttributeError):
        pass

    return result


def enrich_with_api_details(datasets: list[dict]) -> list[dict]:
    """Enrich dataset records with API dataset IDs and publication IDs.

    Visits each dataset's detail page to extract API-specific identifiers
    from the ``__NEXT_DATA__`` JSON. Pauses 0.3 s between requests.

    Args:
        datasets: List of dataset dicts from ``scrape_catalogue()``.

    Returns:
        The same list with ``api_dataset_id`` and ``publication_id`` added.
    """
    total = len(datasets)
    resolved = 0

    print(f"\nEnriching {total} datasets with API detail pages...")
    for i, ds in enumerate(datasets, 1):
        time.sleep(0.3)
        try:
            details = fetch_api_details(ds["dataset_id"])
            ds["api_dataset_id"] = details["api_dataset_id"]
            ds["publication_id"] = details["publication_id"]
            if details["api_dataset_id"]:
                resolved += 1
        except requests.RequestException as e:
            print(f"  WARNING: Failed to fetch detail page for {ds['dataset_id']}: {e}")
            ds["api_dataset_id"] = None
            ds["publication_id"] = None

        if i % 10 == 0 or i == total:
            print(f"  Detail pages: {i}/{total} (resolved {resolved} API IDs)")

    unresolved = total - resolved
    if unresolved > 0:
        print(f"\nWARNING: {unresolved} dataset(s) could not be resolved to an API ID.")
    print(f"Enriched {resolved}/{total} datasets with API IDs.")
    return datasets

In [0]:
all_datasets, expected_total = scrape_catalogue()
all_datasets = enrich_with_api_details(all_datasets)

In [0]:
if len(all_datasets) != expected_total:
    raise RuntimeError(
        f"Expected {expected_total} datasets but scraped {len(all_datasets)}"
    )
print(f"PASS: scraped {len(all_datasets)} == expected {expected_total}")

In [0]:
# Date strings from the catalogue use "d MMM yyyy" format (e.g. "5 Mar 2025")
df_all = (
    spark.createDataFrame(all_datasets)
    .withColumn("published", F.to_date("published", "d MMM yyyy"))
    .withColumn("last_updated", F.to_date("last_updated", "d MMM yyyy"))
)

In [0]:
def validate_dataframe(df: DataFrame, expected_total: int) -> None:
    """Run integrity checks on the scraped DataFrame.

    Checks performed:
        1. DataFrame is non-empty
        2. Row count matches the expected total from the page header
        3. Row count exceeds the safety floor (MIN_ROW_COUNT)
        4. All required columns are present
        5. Critical columns contain no nulls
        6. dataset_id values are unique

    Args:
        df: Scraped catalogue DataFrame to validate.
        expected_total: Dataset count reported by the catalogue page header.

    Raises:
        RuntimeError: If any integrity check fails (merge is aborted).
    """
    row_count = df.count()
    errors = []

    if row_count == 0:
        errors.append("DataFrame is empty - nothing was scraped.")

    if row_count != expected_total:
        errors.append(
            f"Row count ({row_count}) does not match expected total ({expected_total})."
        )

    if row_count < MIN_ROW_COUNT:
        errors.append(
            f"Row count ({row_count}) is suspiciously low (minimum: {MIN_ROW_COUNT})."
        )

    missing_cols = REQUIRED_COLUMNS - set(df.columns)
    if missing_cols:
        errors.append(f"Missing columns: {missing_cols}")

    if not missing_cols:
        null_counts = df.select(
            *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in NON_NULL_COLUMNS]
        ).collect()[0]
        cols_with_nulls = {c: null_counts[c] for c in NON_NULL_COLUMNS if null_counts[c] > 0}
        if cols_with_nulls:
            errors.append(f"Null values found in critical columns: {cols_with_nulls}")

    if "dataset_id" in df.columns:
        distinct_ids = df.select("dataset_id").distinct().count()
        if distinct_ids != row_count:
            errors.append(
                f"Duplicate dataset_ids detected: {row_count} rows but {distinct_ids} distinct IDs."
            )

    if errors:
        error_msg = "Data integrity checks FAILED \u2013 merge aborted:\n  \u2022 " + "\n  \u2022 ".join(errors)
        raise RuntimeError(error_msg)

    print(f"All integrity checks passed ({row_count} rows, {len(df.columns)} columns).")


validate_dataframe(df_all, expected_total)

In [0]:
def _compute_diff(df_source: DataFrame, df_target: DataFrame) -> dict:
    """Compute and print a change report between source and target DataFrames.

    Compares on ``dataset_id`` and detects added, removed, and updated rows
    by checking all non-key columns with null-safe equality.
    Only compares columns present in both source and target (handles schema evolution).

    Args:
        df_source: Freshly scraped catalogue DataFrame.
        df_target: Current contents of the Delta target table.

    Returns:
        Dict with keys ``added``, ``removed``, ``updated``, each holding
        a list of row dicts.
    """
    # Only compare columns that exist in both source and target
    common_cols = set(df_source.columns) & set(df_target.columns)
    compare_cols = [c for c in df_source.columns if c != "dataset_id" and c in common_cols]
    new_cols = [c for c in df_source.columns if c not in df_target.columns]
    if new_cols:
        print(f"New columns in source (will be added): {new_cols}")

    # New datasets (in source but not in target)
    df_added = df_source.join(df_target, on="dataset_id", how="left_anti")
    added_rows = [row.asDict() for row in df_added.collect()]

    # Removed datasets (in target but not in source)
    df_removed = df_target.join(df_source, on="dataset_id", how="left_anti")
    removed_rows = [row.asDict() for row in df_removed.collect()]

    # Updated datasets (matched on dataset_id but differ on any other column)
    df_matched = df_source.alias("s").join(df_target.alias("t"), on="dataset_id", how="inner")
    change_condition = F.lit(False)
    for col in compare_cols:
        change_condition = change_condition | (
            ~F.col(f"s.{col}").eqNullSafe(F.col(f"t.{col}"))
        )
    # If there are new columns, treat all matched rows as updated
    if new_cols:
        change_condition = change_condition | F.lit(True)
    df_updated = df_matched.filter(change_condition).select(
        F.col("dataset_id"),
        *[F.col(f"s.{c}").alias(c) for c in df_source.columns if c != "dataset_id"],
    )
    updated_rows = [row.asDict() for row in df_updated.collect()]

    unchanged_count = df_source.count() - len(added_rows) - len(updated_rows)

    # ---- Print report ----
    print(f"\n\u2795 ADDED: {len(added_rows)} dataset(s)")
    for r in added_rows:
        print(f"    \u2022 {r['dataset_name']}  [{r['publication']}]")

    print(f"\n\u274c REMOVED: {len(removed_rows)} dataset(s)")
    for r in removed_rows:
        print(f"    \u2022 {r['dataset_name']}  [{r['publication']}]")

    print(f"\n\u270f\ufe0f  UPDATED: {len(updated_rows)} dataset(s)")
    for r in updated_rows:
        print(f"    \u2022 {r['dataset_name']}  [{r['publication']}]")

    print(f"\n\u2714\ufe0f  UNCHANGED: {unchanged_count} dataset(s)")

    return {"added": added_rows, "removed": removed_rows, "updated": updated_rows}


def merge_to_delta(df_source: DataFrame, target_table: str) -> dict:
    """Create or merge-upsert the source DataFrame into the target Delta table.

    On first run (table does not exist), the table is created directly.
    On subsequent runs, a MERGE statement handles inserts, updates, and deletes.
    If no changes are detected, the merge is skipped entirely.

    Args:
        df_source: Validated catalogue DataFrame to write.
        target_table: Fully qualified Delta table name.

    Returns:
        Dict with keys ``added``, ``removed``, ``updated``, each holding
        a list of row dicts describing what changed.
    """
    row_count = df_source.count()
    print("\n", "=" * 60)

    if not spark.catalog.tableExists(target_table):
        print(f"Target table does not exist \u2013 creating {target_table}")
        df_source.write.saveAsTable(target_table)

        added_rows = [row.asDict() for row in df_source.collect()]
        print(f"\n\u2795 ADDED: {len(added_rows)} dataset(s) (initial load)")
        for r in added_rows:
            print(f"    \u2022 {r['dataset_name']}  [{r['publication']}]")

        changes = {"added": added_rows, "removed": [], "updated": []}
    else:
        df_target = spark.table(target_table)
        changes = _compute_diff(df_source, df_target)

        if not any([changes["added"], changes["removed"], changes["updated"]]):
            print("\nNo changes detected \u2013 skipping merge.")
            return changes

        print("\n", "=" * 60)

        df_source.createOrReplaceTempView("source_catalogue")
        spark.sql(f"""
            MERGE INTO {target_table} AS target
            USING source_catalogue AS source
            ON target.dataset_id = source.dataset_id
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED BY TARGET THEN INSERT *
            WHEN NOT MATCHED BY SOURCE THEN DELETE
        """)
        spark.catalog.dropTempView("source_catalogue")

    print(f"\nSuccessfully merged {row_count} rows into {target_table}")
    return changes


catalogue_changes = merge_to_delta(df_all, TARGET_TABLE)